# German gender/case — step-by-step workflow

This notebook walks through the GRADIEND workflow for German definite articles (der/die/das, etc.) in a **step-by-step** way. We start with a **single** gender–case pair, understand each part, then show how to scale to many pairs and compare them.

**What you'll do:**
1. Train one GRADIEND model for one pair (e.g. masculine vs feminine nominative)
2. Plot training convergence and encoder distributions
3. Run decoder evaluation and optionally obtain a changed model
4. Optionally: loop over many pairs and visualize top-k overlap across runs

**See also:** [gender_de_detailed.py](gender_de_detailed.py) — the same workflow as a script (starts with the loop).

## 1. Setup — one pair and training arguments

We choose **one** target pair: masculine nominative vs feminine nominative (masc_nom / fem_nom). This is the simplest entry point: the model learns to separate these two classes from their gradient differences.

We use precomputed Hugging Face data (`de-gender-case-articles`) and neutral data for evaluation. Training arguments include **pre-pruning** (1% top weights) and **post-pruning** (5%) to reduce memory and speed up training—see the [Pruning guide](https://aieng-lab.github.io/gradiend/guides/pruning-guide/).

In [ ]:
from gradiend import TextPredictionTrainer, TrainingArguments
from gradiend.trainer import PrePruneConfig, PostPruneConfig

model_name = "bert-base-german-cased"
pair = ("masc_nom", "fem_nom")

args = TrainingArguments(
    experiment_dir="runs/notebook_gender_de_detailed",
    train_batch_size=8,
    train_max_size=500,
    encoder_eval_max_size=50,
    decoder_eval_max_size_training_like=100,
    decoder_eval_max_size_neutral=500,
    eval_steps=100,
    num_train_epochs=1,
    max_seeds=3,
    min_convergent_seeds=1,
    max_steps=100,
    source="alternative",
    target="diff",
    eval_batch_size=8,
    learning_rate=5e-5,
    pre_prune_config=PrePruneConfig(n_samples=16, topk=0.01, source="diff"),
    post_prune_config=PostPruneConfig(topk=0.05, part="decoder-weight"),
    use_cache=True,
    add_identity_for_other_classes=True,
)

trainer = TextPredictionTrainer(
    model=model_name,
    run_id=f"gender_de_{pair[0]}_{pair[1]}",
    data="aieng-lab/de-gender-case-articles",
    target_classes=list(pair),
    masked_col="masked",
    split_col="split",
    eval_neutral_data="aieng-lab/wortschatz-leipzig-de-grammar-neutral_data",
    args=args,
)

## 2. Train and plot convergence

`trainer.train()` runs GRADIEND training. With `use_cache=True`, it will skip if a model already exists. After training, `plot_training_convergence()` shows correlation and mean-by-class over steps—high correlation means the encoder learned to separate the two classes.

In [ ]:
trainer.train()
stats = trainer.get_training_stats()
ts = stats.get("training_stats", {}) if stats else {}
print(f"Correlation: {ts.get('correlation')}, mean_by_class: {ts.get('mean_by_class')}")

ENCODER_LABEL_NAMES = {
    "masc_nom": "Masc. Nom.", "fem_nom": "Fem. Nom.",
    "masc_acc": "Masc. Acc.", "fem_acc": "Fem. Acc.",
    "masc_dat": "Masc. Dat.", "fem_dat": "Fem. Dat.",
    "masc_gen": "Masc. Gen.", "fem_gen": "Fem. Gen.",
    "neut_nom": "Neut. Nom.", "neut_acc": "Neut. Acc.",
    "neut_dat": "Neut. Dat.", "neut_gen": "Neut. Gen.",
}
trainer.plot_training_convergence(label_name_mapping=ENCODER_LABEL_NAMES)

## 3. Encoder evaluation — distributions by class

Encoder evaluation answers: *Do the encoded gradients separate the feature classes?* We encode data and optionally neutral examples, then compute correlation and a distribution plot. Target classes should encode to opposite poles (around ±1); neutral data should cluster near 0.

In [ ]:
max_size = 100
enc_eval = trainer.evaluate_encoder(max_size=max_size, return_df=True, plot=True)
print("Encoder metrics:", enc_eval.get("correlation"), enc_eval.get("all_data"))

## 4. Decoder evaluation and changed model

Decoder evaluation answers: *Can we shift the base model's behavior?* It tries different learning rates and feature factors, then selects the best config (e.g. by LMS threshold). Use `rewrite_base_model` to obtain a model biased toward one class—in memory only, or save to disk with `output_dir`.

In [ ]:
dec_results = trainer.evaluate_decoder()
print("Decoder summary keys:", list(dec_results["summary"].keys()))
# Uncomment to get a model biased toward masc_nom (or save to disk):
# changed_model = trainer.rewrite_base_model(decoder_results=dec_results, target_class=pair[0], output_dir="./output")

## 5. (Optional) Loop over many pairs and compare

Now that we understand one run, we can scale to many pairs. The script [gender_de_detailed.py](gender_de_detailed.py) trains all pairs in `configs` and then plots **top-k overlap** (Venn diagrams and heatmap) to see which runs modify the same or different parameters.

Run the cell below only if you want to train a few more pairs and compare. Warning: this is slower and uses more GPU memory.

In [ ]:
# Optional: train a few more pairs and plot top-k overlap
import os
from gradiend.visualizer.topk.pairwise_heatmap import plot_topk_overlap_heatmap
from gradiend.visualizer.topk.venn_ import plot_topk_overlap_venn

# A small subset for demonstration (full configs in gender_de_detailed.py)
extra_pairs = [
    ("masc_nom", "fem_nom"),  # already trained
    ("masc_nom", "neut_nom"),
    ("fem_nom", "neut_nom"),
]
models_for_heatmap = {trainer.run_id: trainer.get_model()}

for p in extra_pairs[1:]:  # skip first, already trained
    t = TextPredictionTrainer(
        model=model_name,
        run_id=f"gender_de_{p[0]}_{p[1]}",
        data="aieng-lab/de-gender-case-articles",
        target_classes=list(p),
        masked_col="masked",
        split_col="split",
        eval_neutral_data="aieng-lab/wortschatz-leipzig-de-grammar-neutral_data",
        args=args,
    )
    t.train()
    t.cpu()
    models_for_heatmap[t.run_id] = t.get_model()

# Heatmap of pairwise top-k overlaps
plot_topk_overlap_heatmap(
    models_for_heatmap,
    topk=1000,
    part="decoder-weight",
    value="intersection_frac",
    output_path=os.path.join(args.experiment_dir, "topk_overlap_heatmap.png"),
)

## Next steps

- **[Tutorial: Training](https://aieng-lab.github.io/gradiend/tutorials/training/)** — Pruning, multi-seed, caching, and training options.
- **[Tutorial: Evaluation (intra-model)](https://aieng-lab.github.io/gradiend/tutorials/evaluation-intra-model/)** — Encoder/decoder in depth.
- **[Tutorial: Evaluation (inter-model)](https://aieng-lab.github.io/gradiend/tutorials/evaluation-inter-model/)** — Top-k overlap and heatmaps.